# Declarative vs Imperative IaC — a local sandbox experiment

> L2 concept exercise — Infrastructure as Code. I wanted to feel the difference between the two IaC styles with my own hands instead of just reading about it, so I built the same three-container stack (an nginx web front, a redis cache, and a tiny app) twice on my laptop: once with a string of `docker run` commands, and once from a single `docker-compose.yml`. No cloud account needed — just Docker on the host. This is the sandbox version of the declarative-vs-imperative split the primer called out, and I wanted to see it move.


## Setup

I need Docker Engine and the compose plugin on this machine. The cells below print the versions so I know my sandbox is ready before I start stacking containers — if Docker is missing I stop here rather than chasing a half-built stack.


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker --version
!docker compose version


## The imperative way — one command at a time

Imperative IaC reads like a recipe: do this, then this, then this. Each `docker run` is an explicit instruction, and the running stack exists only as a side effect of the commands I just issued. There is no single file that records the desired end state — if I close this notebook, the state lives only in Docker until I tear it down.


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker network create iac-lab-net
!docker run -d --name iac-lab-web --network iac-lab-net nginx:alpine
!docker run -d --name iac-lab-cache --network iac-lab-net redis:alpine


### What I saw after the imperative run

I have to ask Docker for the state — `docker ps` and `docker network inspect` — because nothing on disk recorded it for me. The stack is real, but only Docker knows its shape, and I know it only because I ran the commands in that order.


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker ps --format "{{.Names}}  {{.Status}}  {{.Ports}}"
!docker network inspect iac-lab-net --format "{{range .Containers}}{{.Name}} {{end}}"


### Tear down

Three `rm`-style commands just to undo what I built. If I missed one, it lingers on the host — the classic drift that starts with "I'll just leave this container running for a minute."


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker rm -f iac-lab-web iac-lab-cache
!docker network rm iac-lab-net


## The declarative way — describe the desired state

Declarative IaC flips the script: I write the end state I want (services, ports, dependencies) in one file, and the tool figures out the steps to reach it. Running the same file again should be a no-op if nothing changed — that is idempotency. `docker compose config` lets me preview the resolved desired state before anything runs, which is the "plan" moment.


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
from pathlib import Path
compose = """\
services:
  web:
    image: nginx:alpine
    ports: ["8080:80"]
    depends_on: [cache]
  cache:
    image: redis:alpine
"""
Path("docker-compose.yml").write_text(compose)
print(Path("docker-compose.yml").read_text())


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker compose -p iac-lab config


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker compose -p iac-lab up -d


### What I saw after the declarative run

Same two containers, same shared network — but now `docker compose ps` and the `docker-compose.yml` sitting on disk agree with each other. If I hand that file to a teammate, they get the same stack from one command, and a code review of the file is literally a review of the infrastructure.


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker compose -p iac-lab ps


### Tear down

One command, and the tool removes everything it created — networks, containers, the lot. No manual `rm` per resource, and nothing left behind to drift.


In [ ]:
# last_verified: 2026-08-08 · IaC concepts n/a
!docker compose -p iac-lab down


## Comparing the two

| Dimension | Imperative (`docker run`) | Declarative (`docker-compose.yml`) |
|---|---|---|
| Source of truth | My memory, or this notebook's history | `docker-compose.yml`, checkable into Git |
| Reproduce elsewhere | Re-run every command, in order | `docker compose up -d` |
| Drift correction | Never — only I remember what I changed | `docker compose up` reconciles back to the file |
| Partial failure | Stack left half-built | No update until the file is valid |
| Review a change | Scrolled through shell sessions | One diff per changed file |

The table makes it plain: imperative is fast to type and easy to read line-by-line, but it stores the desired state in my head. Declarative takes longer to write up front, but the file becomes the contract — the thing the tool and my teammates and CI all agree on.


## What tripped me up

- **Imperative order matters.** I almost started the web container before creating the network, and it failed with "network not found." With declarative YAML I just declare both services and the tool resolves the graph — `depends_on: [cache]` is the lever I use to express ordering without enforcing it by hand.
- **Imperative drift is silent.** After I tweaked a `docker run` flag mid-debug, the next run with my old flags would not have matched the live container. With compose, any change to `image:` or `ports:` must go through the file and a fresh `docker compose up` — drift has to pass through the desired-state document.
- **Names and networks differ.** The imperative run used my hand-named `iac-lab-net`; compose created its own project network. For local practice the names line up, but in the cloud these map to VPCs and subnets — same idea, just bigger blast radius.


## What I'd try next

I want to redo this same comparison with a real provisioner: write a small `main.tf` (declarative) and let the `docker run` commands above stand in for the imperative script that an old setup.sh would run. Running `terraform plan` then `terraform apply` against the Docker provider locally should let me see plan-then-apply and drift detection without touching a cloud account — the exercise ideas from research mention exactly this kind of local practice loop.


## Sources

I drew on the same Infrastructure as Code research that backs this concept. Each URL is a verbatim research source, not an invented reference.
- https://trungtmnguyen.com/en/blog/common-terraform-mistakes-and-how-to-avoid-them — Terraform pitfalls and state
- https://devops-daily.com/posts/infrastructure-as-code-fundamentals — IaC fundamentals overview
- https://www.hyaking.com/infrastructure-as-code-terraform-beginners-guide/ — beginner Terraform guide
- https://teachmeidea.com/infrastructure-as-code-with-terraform-beginner-to-pro/ — Terraform beginner to pro
